In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
import gc

In [2]:
gc.collect()
df1 = pd.read_csv("./ev_battery_failure_dataset.csv").copy().reset_index(drop = True)
df1

,vehicle_id,vehicle_brand,vehicle_model,vehicle_type,manufacturing_year,battery_manufacturer,battery_chemistry,battery_capacity_kwh,drive_type,odometer_km,...,sensor_fault_count,BMS_warning_count,abnormal_voltage_events,battery_stress_index,aging_score,thermal_health_score,charging_quality_score,driving_stress_score,predicted_remaining_life_cycles,battery_failure
0,EV100000,Nissan,Leaf,SUV,2019.0,CATL,LMO,82.54,RWD,98163.0,...,2.0,3.0,0.0,0.0,53.5,66.0,65.2,0.0,401.0,0
1,EV100001,Audi,e-tron,Sedan,2018.0,Samsung SDI,NaN,64.17,AWD,113600.0,...,0.0,3.0,0.0,51.0,38.8,67.8,46.3,40.9,1094.0,0
2,EV100002,Toyota,bZ4X,Van,2018.0,Guoxuan,NMC,90.92,RWD,242253.0,...,4.0,1.0,0.0,36.2,66.1,48.6,0.0,6.5,645.0,0
3,EV100003,Volkswagen,ID.3,Hatchback,2022.0,CATL,NMC,47.35,AWD,83700.0,...,0.0,3.0,1.0,23.4,32.5,58.2,NaN,NaN,867.0,0
4,EV100004,Tesla,Model X,Crossover,2015.0,BYD Battery,NMC,68.04,FWD,88784.0,...,3.0,2.0,1.0,51.2,45.4,51.0,NaN,100.0,1090.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,EV299995,BYD,Song Plus,Hatchback,2016.0,CATL,LFP,45.99,FWD,182311.0,...,3.0,1.0,1.0,62.3,NaN,78.6,0.0,51.5,2691.0,0
199996,EV299996,Lucid,Air,Crossover,2015.0,LG Energy Solution,NMC,67.65,AWD,275317.0,...,2.0,5.0,2.0,33.1,89.5,59.3,73.7,57.0,420.0,0
199997,EV299997,BYD,Song Plus,SUV,NaN,BYD Battery,LFP,98.08,FWD,158747.0,...,2.0,2.0,0.0,79.0,13.1,64.7,32.8,7.6,3148.0,0
199998,EV299998,Tesla,Model Y,Hatchback,2016.0,SK On,NMC,51.49,FWD,95122.0,...,2.0,2.0,0.0,24.3,59.5,50.6,69.4,12.7,797.0,0


In [3]:
class KNearestNeighbours(object):
    def __init__(self, dataset, n_neighbours : int, weight : str, algorithm : str, leaf_size : int, p : int, 
                metric : str, metric_params : dict, n_jobs : int):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.n_neighbours = n_neighbours
        self.weight = weight
        self.algorithm = algorithm
        self.leaf_size = leaf_size
        self.p = p
        self.metric = metric
        self.metric_params = metric_params
        self.n_jobs = n_jobs

    def data_preprocess(self):
        global df3_global
        df2 = self.dataset.copy().reset_index(drop = True)
        df2 = df2.dropna(how='any',axis=0)
        df2.loc[df2['vehicle_brand'] == 'Audi', 'Vehicle_Brand_Num_Value'] = '0'
        df2.loc[df2['vehicle_brand'] == 'BMW', 'Vehicle_Brand_Num_Value'] = '01'
        df2.loc[df2['vehicle_brand'] == 'BYD', 'Vehicle_Brand_Num_Value'] = '02'
        df2.loc[df2['vehicle_brand'] == 'Ford', 'Vehicle_Brand_Num_Value'] = '03'
        df2.loc[df2['vehicle_brand'] == 'GM', 'Vehicle_Brand_Num_Value'] = '04'
        df2.loc[df2['vehicle_brand'] == 'Honda', 'Vehicle_Brand_Num_Value'] = '05'
        df2.loc[df2['vehicle_brand'] == 'Hyundai', 'Vehicle_Brand_Num_Value'] = '06'
        df2.loc[df2['vehicle_brand'] == 'Kia', 'Vehicle_Brand_Num_Value'] = '07'
        df2.loc[df2['vehicle_brand'] == 'Lucid', 'Vehicle_Brand_Num_Value'] = '08'
        df2.loc[df2['vehicle_brand'] == 'MG', 'Vehicle_Brand_Num_Value'] = '09'
        df2.loc[df2['vehicle_brand'] == 'Mahindra', 'Vehicle_Brand_Num_Value'] = '10'
        df2.loc[df2['vehicle_brand'] == 'Mercedes', 'Vehicle_Brand_Num_Value'] = '11'
        df2.loc[df2['vehicle_brand'] == 'Nissan', 'Vehicle_Brand_Num_Value'] = '12'
        df2.loc[df2['vehicle_brand'] == 'Renault', 'Vehicle_Brand_Num_Value'] = '13'
        df2.loc[df2['vehicle_brand'] == 'Rivian', 'Vehicle_Brand_Num_Value'] = '14'
        df2.loc[df2['vehicle_brand'] == 'Tata', 'Vehicle_Brand_Num_Value'] = '15'
        df2.loc[df2['vehicle_brand'] == 'Tesla', 'Vehicle_Brand_Num_Value'] = '16'
        df2.loc[df2['vehicle_brand'] == 'Toyota', 'Vehicle_Brand_Num_Value'] = '17'
        df2.loc[df2['vehicle_brand'] == 'Volkswagen', 'Vehicle_Brand_Num_Value'] = '18'
        df2.loc[df2['vehicle_brand'] == 'Volvo', 'Vehicle_Brand_Num_Value'] = '19'
        df2.loc[df2['vehicle_model'] == 'Air', 'Vehicle_Model_Num_Value'] = '0'
        df2.loc[df2['vehicle_model'] == 'Ariya', 'Vehicle_Model_Num_Value'] = '01'
        df2.loc[df2['vehicle_model'] == 'Atto 3', 'Vehicle_Model_Num_Value'] = '02'
        df2.loc[df2['vehicle_model'] == 'BE 6', 'Vehicle_Model_Num_Value'] = '03'
        df2.loc[df2['vehicle_model'] == 'Cadillac Lyriq', 'Vehicle_Model_Num_Value'] = '04'
        df2.loc[df2['vehicle_model'] == 'Chevrolet Bolt', 'Vehicle_Model_Num_Value'] = '05'
        df2.loc[df2['vehicle_model'] == 'Comet EV', 'Vehicle_Model_Num_Value'] = '06'
        df2.loc[df2['vehicle_model'] == 'Dolphin', 'Vehicle_Model_Num_Value'] = '07'
        df2.loc[df2['vehicle_model'] == 'EQB', 'Vehicle_Model_Num_Value'] = '08'
        df2.loc[df2['vehicle_model'] == 'EQE', 'Vehicle_Model_Num_Value'] = '09'
        df2.loc[df2['vehicle_model'] == 'EQS', 'Vehicle_Model_Num_Value'] = '10'
        df2.loc[df2['vehicle_model'] == 'EV6', 'Vehicle_Model_Num_Value'] = '11'
        df2.loc[df2['vehicle_model'] == 'EV9', 'Vehicle_Model_Num_Value'] = '12'
        df2.loc[df2['vehicle_model'] == 'EX30', 'Vehicle_Model_Num_Value'] = '13'
        df2.loc[df2['vehicle_model'] == 'EX90', 'Vehicle_Model_Num_Value'] = '14'
        df2.loc[df2['vehicle_model'] == 'F-150 Lightning', 'Vehicle_Model_Num_Value'] = '15'
        df2.loc[df2['vehicle_model'] == 'Gravity', 'Vehicle_Model_Num_Value'] = '16'
        df2.loc[df2['vehicle_model'] == 'Han', 'Vehicle_Model_Num_Value'] = '17'
        df2.loc[df2['vehicle_model'] == 'ID.3', 'Vehicle_Model_Num_Value'] = '18'
        df2.loc[df2['vehicle_model'] == 'ID.4', 'Vehicle_Model_Num_Value'] = '19'
        df2.loc[df2['vehicle_model'] == 'ID.Buzz', 'Vehicle_Model_Num_Value'] = '20'
        df2.loc[df2['vehicle_model'] == 'Ioniq 5', 'Vehicle_Model_Num_Value'] = '21'
        df2.loc[df2['vehicle_model'] == 'Ioniq 6', 'Vehicle_Model_Num_Value'] = '22'
        df2.loc[df2['vehicle_model'] == 'Kona Electric', 'Vehicle_Model_Num_Value'] = '23'
        df2.loc[df2['vehicle_model'] == 'Leaf', 'Vehicle_Model_Num_Value'] = '24'
        df2.loc[df2['vehicle_model'] == 'MG4', 'Vehicle_Model_Num_Value'] = '25'
        df2.loc[df2['vehicle_model'] == 'Megane E-Tech', 'Vehicle_Model_Num_Value'] = '26'
        df2.loc[df2['vehicle_model'] == 'Model 3', 'Vehicle_Model_Num_Value'] = '27'
        df2.loc[df2['vehicle_model'] == 'Model S', 'Vehicle_Model_Num_Value'] = '28'
        df2.loc[df2['vehicle_model'] == 'Model X', 'Vehicle_Model_Num_Value'] = '29'
        df2.loc[df2['vehicle_model'] == 'Model Y', 'Vehicle_Model_Num_Value'] = '30'
        df2.loc[df2['vehicle_model'] == 'Mustang Mach-E', 'Vehicle_Model_Num_Value'] = '31'
        df2.loc[df2['vehicle_model'] == 'Nexon EV', 'Vehicle_Model_Num_Value'] = '32'
        df2.loc[df2['vehicle_model'] == 'Niro EV', 'Vehicle_Model_Num_Value'] = '33'
        df2.loc[df2['vehicle_model'] == 'Prologue', 'Vehicle_Model_Num_Value'] = '34'
        df2.loc[df2['vehicle_model'] == 'Punch EV', 'Vehicle_Model_Num_Value'] = '35'
        df2.loc[df2['vehicle_model'] == 'Q4 e-tron', 'Vehicle_Model_Num_Value'] = '36'
        df2.loc[df2['vehicle_model'] == 'R1S', 'Vehicle_Model_Num_Value'] = '37'
        df2.loc[df2['vehicle_model'] == 'R1T', 'Vehicle_Model_Num_Value'] = '38'
        df2.loc[df2['vehicle_model'] == 'Silverado EV', 'Vehicle_Model_Num_Value'] = '39'
        df2.loc[df2['vehicle_model'] == 'Song Plus', 'Vehicle_Model_Num_Value'] = '40'
        df2.loc[df2['vehicle_model'] == 'Tang', 'Vehicle_Model_Num_Value'] = '41'
        df2.loc[df2['vehicle_model'] == 'Tiago EV', 'Vehicle_Model_Num_Value'] = '42'
        df2.loc[df2['vehicle_model'] == 'XC40 Recharge', 'Vehicle_Model_Num_Value'] = '43'
        df2.loc[df2['vehicle_model'] == 'XUV400', 'Vehicle_Model_Num_Value'] = '44'
        df2.loc[df2['vehicle_model'] == 'ZS EV', 'Vehicle_Model_Num_Value'] = '45'
        df2.loc[df2['vehicle_model'] == 'Zoe', 'Vehicle_Model_Num_Value'] = '46'
        df2.loc[df2['vehicle_model'] == 'bZ3', 'Vehicle_Model_Num_Value'] = '47'
        df2.loc[df2['vehicle_model'] == 'bZ4X', 'Vehicle_Model_Num_Value'] = '48'
        df2.loc[df2['vehicle_model'] == 'e-tron', 'Vehicle_Model_Num_Value'] = '49'
        df2.loc[df2['vehicle_model'] == 'e-tron GT', 'Vehicle_Model_Num_Value'] = '50'
        df2.loc[df2['vehicle_model'] == 'e:Ny1', 'Vehicle_Model_Num_Value'] = '51'
        df2.loc[df2['vehicle_model'] == 'i4', 'Vehicle_Model_Num_Value'] = '52'
        df2.loc[df2['vehicle_model'] == 'i7', 'Vehicle_Model_Num_Value'] = '53'
        df2.loc[df2['vehicle_model'] == 'iX', 'Vehicle_Model_Num_Value'] = '54'
        df2.loc[df2['battery_manufacturer'] == 'BYD Battery', 'Battery_Manufacturer_Num_Value'] = '0'
        df2.loc[df2['battery_manufacturer'] == 'CATL', 'Battery_Manufacturer_Num_Value'] = '01'
        df2.loc[df2['battery_manufacturer'] == 'Envision AESC', 'Battery_Manufacturer_Num_Value'] = '02'
        df2.loc[df2['battery_manufacturer'] == 'Guoxuan', 'Battery_Manufacturer_Num_Value'] = '03'
        df2.loc[df2['battery_manufacturer'] == 'LG Energy Solution', 'Battery_Manufacturer_Num_Value'] = '04'
        df2.loc[df2['battery_manufacturer'] == 'Northvolt', 'Battery_Manufacturer_Num_Value'] = '05'
        df2.loc[df2['battery_manufacturer'] == 'Panasonic', 'Battery_Manufacturer_Num_Value'] = '06'
        df2.loc[df2['battery_manufacturer'] == 'SK On', 'Battery_Manufacturer_Num_Value'] = '07'
        df2.loc[df2['battery_manufacturer'] == 'Samsung SDI', 'Battery_Manufacturer_Num_Value'] = '08'
        df2['Vehicle_Brand_Num_Value'] = df2['Vehicle_Brand_Num_Value'].astype(int)
        df2['Vehicle_Model_Num_Value'] = df2['Vehicle_Model_Num_Value'].astype(int)
        df2['Battery_Manufacturer_Num_Value'] = df2['Battery_Manufacturer_Num_Value'].astype(int)
        df2['manufacturing_year'] = df2['manufacturing_year'].astype(int)
        numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
        df2 = df2.select_dtypes(include=numerics)
        df3_global = df2.copy().reset_index(drop = True)

    def train_split(self):

        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        from sklearn.model_selection import train_test_split
        df4 = df3_global.copy().reset_index(drop = True)
        Y_col = 'Battery_Manufacturer_Num_Value'
        X_cols = df4.loc[:, df4.columns != Y_col].columns
        # Split into 80% training and 20% testing
        X_train, X_test, y_train, y_test = train_test_split(
            df3_global[X_cols], df3_global[Y_col], test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def ML_fitting(self):
        global knn_fit_global

        knn = KNeighborsClassifier(n_neighbors=self.n_neighbours, weights=self.weight, algorithm=self.algorithm, 
                                   leaf_size=self.leaf_size, p=self.p, 
                                   metric=self.metric, metric_params=self.metric_params, n_jobs=self.n_jobs)
        knn_fit = knn.fit(X_train_global, y_train_global)
        knn_fit_global = knn_fit

    def ML_prediction(self):
        global knn_y_predict_global

        gc.collect()
        knn_y_predict = knn_fit_global.predict(X_test_global)
        knn_y_predict_global = knn_y_predict

    def evaluation(self):

        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(y_test_global, knn_y_predict_global)
        print("Accuracy:", accuracy)

        from sklearn.metrics import balanced_accuracy_score
        balanced_accuracy_score = balanced_accuracy_score(y_test_global, knn_y_predict_global)
        print("balanced_accuracy_score:", balanced_accuracy_score)

        from sklearn.metrics import top_k_accuracy_score
        top_k_accuracy_score = top_k_accuracy_score(y_test_global,knn_fit_global.predict_proba(X_test_global), k = 1)
        print("top_k_accuracy_score:", top_k_accuracy_score)

        from sklearn.metrics import average_precision_score
        average_precision_score = average_precision_score(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("average_precision_score:", average_precision_score)

        from sklearn.metrics import brier_score_loss
        brier_score_loss = brier_score_loss(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("brier_score_loss:", brier_score_loss)

        from sklearn.metrics import f1_score
        f1_score = f1_score(y_test_global, knn_y_predict_global, average='macro')
        print("f1_score_macro:", f1_score)
        from sklearn.metrics import f1_score
        f1_score_1 = f1_score(y_test_global, knn_y_predict_global, average='micro')
        print("f1_score_micro:", f1_score_1)
        f1_score_2 = f1_score(y_test_global, knn_y_predict_global, average='weighted')
        print("f1_score_weighted:", f1_score_2)
        f1_score_3 = f1_score(y_test_global, knn_y_predict_global, average=None)
        print("f1_score_None:", f1_score_3)

        from sklearn.metrics import log_loss
        log_loss = log_loss(y_test_global,knn_fit_global.predict_proba(X_test_global))
        print("log_loss:", log_loss)

        from sklearn.metrics import precision_score
        precision_score_macro = precision_score(y_test_global, knn_y_predict_global, average='macro')
        print("precision_score_macro:", precision_score_macro)
        precision_score_micro = precision_score(y_test_global, knn_y_predict_global, average='micro')
        print("precision_score_micro:", precision_score_micro)
        precision_score_weighted = precision_score(y_test_global, knn_y_predict_global, average='weighted')
        print("precision_score_weighted:", precision_score_weighted)
        precision_score_None = precision_score(y_test_global, knn_y_predict_global, average=None)
        print("precision_score_None:", precision_score_None)
        precision_score_None_ZD_1 = precision_score(y_test_global, knn_y_predict_global, average=None, zero_division=1)
        print("precision_score_None_ZD_1:", precision_score_None_ZD_1)
        precision_score_None_ZD_NAN = precision_score(y_test_global, knn_y_predict_global, average=None, zero_division=np.nan)
        print("precision_score_None_ZD_NAN:", precision_score_None_ZD_NAN)

        from sklearn.metrics import recall_score
        recall_score_macro = recall_score(y_test_global, knn_y_predict_global, average='macro')
        print("recall_score_macro:", recall_score_macro)
        recall_score_micro = recall_score(y_test_global, knn_y_predict_global, average='micro')
        print("recall_score_micro:", recall_score_micro)
        recall_score_weighted = recall_score(y_test_global, knn_y_predict_global, average='weighted')
        print("recall_score_weighted:", recall_score_weighted)
        recall_score_None = recall_score(y_test_global, knn_y_predict_global, average=None)
        print("recall_score_None:", recall_score_None)
        recall_score_None_ZD_1 = recall_score(y_test_global, knn_y_predict_global, average=None, zero_division=1)
        print("recall_score_None_ZD_1:", recall_score_None_ZD_1)
        recall_score_None_ZD_NAN = recall_score(y_test_global, knn_y_predict_global, average=None, zero_division=np.nan)
        print("recall_score_None_ZD_NAN:", recall_score_None_ZD_NAN)

        from sklearn.metrics import jaccard_score
        jaccard_score_macro = jaccard_score(y_test_global, knn_y_predict_global, average='macro')
        print("jaccard_score_macro:", jaccard_score_macro)
        jaccard_score_micro = jaccard_score(y_test_global, knn_y_predict_global, average='micro')
        print("jaccard_score_micro:", jaccard_score_micro)
        jaccard_score_weighted = jaccard_score(y_test_global, knn_y_predict_global, average='weighted')
        print("jaccard_score_weighted:", jaccard_score_weighted)
        jaccard_score_None = jaccard_score(y_test_global, knn_y_predict_global, average=None)
        print("jaccard_score_None:", jaccard_score_None)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), multi_class = 'ovr')
        print('roc_auc_score_ovr:', roc_auc_score_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), multi_class = 'ovo')
        print('roc_auc_score_ovo:', roc_auc_score_ovo)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovr')
        print("roc_auc_score_macro_ovr:", roc_auc_score_macro_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovo')
        print("roc_auc_score_macro_ovo:", roc_auc_score_macro_ovo)
        
        roc_auc_score_weighted_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovr')
        print("roc_auc_score_weighted_ovr:", roc_auc_score_weighted_ovr)
        roc_auc_score_weighted_ovo = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovo')
        print("roc_auc_score_weighted_ovo:", roc_auc_score_weighted_ovo)
        
        roc_auc_score_None_ovr = roc_auc_score(y_test_global, knn_fit_global.predict_proba(X_test_global), average=None,
                                              multi_class = 'ovr')
        print("roc_auc_score_None_ovr:", roc_auc_score_None_ovr)

        from sklearn.metrics import d2_log_loss_score
        d2_log_loss_score = d2_log_loss_score(y_test_global, knn_fit_global.predict_proba(X_test_global),
                                              sample_weight=None, labels=None)
        print("d2_log_loss_score:", d2_log_loss_score)

        from sklearn.metrics import d2_brier_score
        d2_brier_score = d2_brier_score(y_test_global, knn_fit_global.predict_proba(X_test_global), 
                                        sample_weight=None, labels=None)
        print("d2_brier_score:", d2_brier_score)
        

    def main(self):
        gc.collect()
        self.data_preprocess()
        self.train_split()
        self.ML_fitting()
        self.ML_prediction()
        self.evaluation()
        gc.collect()
        

In [4]:
knn = KNearestNeighbours(df1, 5, 'uniform', 'auto', 30, 2, 'minkowski', None, None)
knn.main()

Accuracy: 0.14777403666292555
balanced_accuracy_score: 0.1083705431971915
top_k_accuracy_score: 0.13168724279835392
average_precision_score: 0.11202620379185843
brier_score_loss: 1.0401496445940892
f1_score_macro: 0.10156625944070878
f1_score_micro: 0.14777403666292555
f1_score_weighted: 0.13176865410942643
f1_score_None: [0.14623656 0.26174497 0.08086253 0.03921569 0.15658363 0.02690583
 0.07920792 0.06806283 0.05527638]
log_loss: 18.853243347474507
precision_score_macro: 0.10843232972999678
precision_score_micro: 0.14777403666292555
precision_score_weighted: 0.13046191127833787
precision_score_None: [0.11038961 0.21690768 0.0887574  0.0625     0.165      0.04615385
 0.0990099  0.10569106 0.08148148]
precision_score_None_ZD_1: [0.11038961 0.21690768 0.0887574  0.0625     0.165      0.04615385
 0.0990099  0.10569106 0.08148148]
precision_score_None_ZD_NAN: [0.11038961 0.21690768 0.0887574  0.0625     0.165      0.04615385
 0.0990099  0.10569106 0.08148148]
recall_score_macro: 0.1083705

In [5]:
knn_fit_global

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](9,)","[0,1,2,...,6,7,8]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
